# 03 · SD vs NS statistics

Group-level comparison of resting-state band power after sleep deprivation (SD) vs normal sleep (NS) in ds004902. **Eyes open (EO) is the primary analysis; eyes closed (EC) is secondary** and uses only the participants who have it. EO and EC are never pooled.

All computations live in `analysis/` (this notebook only calls them and shows tables). Inputs come from the batch pipeline (`python -m batch.run_batch`). These are group-level spectral differences only; no clinical interpretation is implied.

In [1]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT))

import pandas as pd
from analysis.regions import REGIONS
from analysis.stats import run_all

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda v: f'{v:.4g}')

## Run every analysis

Writes `results/features_region.csv`, `results/stats_*.csv` and `results/cluster_results.csv`. The per-electrode cluster test uses 5000 sign-flip permutations (~20 s).

In [2]:
tables = run_all(PROJECT / 'results', PROJECT / 'data', n_permutations=5000)
{name: len(table) for name, table in tables.items()}

Could not find a adjacency matrix for the data. Computing adjacency based on Delaunay triangulations.


-- number of adjacent vertices : 61


C:\Users\Amalia\Documents\ITBA (post formateo)\PSIB\PSIB2026Q2-TPS-Grupo11-Vleugels_Moro_Pezzano\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'features_region': 4120,
 'stats_region_log_power': 40,
 'stats_region_rel_power': 40,
 'stats_frontal_theta_beta': 2,
 'stats_session_order': 42,
 'stats_epochs': 4,
 'stats_participants': 109,
 'cluster_results': 24,
 'stats_cluster_channels': 488}

## Region mapping

Built from the channel names in the BIDS `channels.tsv` (see `analysis/regions.py`).

In [3]:
pd.DataFrame([(r, len(c), ', '.join(c)) for r, c in REGIONS.items()],
             columns=['region', 'n_channels', 'channels'])

,region,n_channels,channels
0,frontal,16,"Fp1, Fpz, Fp2, AF3, AF4, AF7, AF8, Fz, F1, F2,..."
1,central,20,"FC1, FC2, FC3, FC4, FC5, FC6, Cz, C1, C2, C3, ..."
2,temporal,8,"FT7, FT8, T7, T8, TP7, TP8, TP9, TP10"
3,parietal,9,"Pz, P1, P2, P3, P4, P5, P6, P7, P8"
4,occipital,8,"POz, PO3, PO4, PO7, PO8, Oz, O1, O2"


## Participants and epochs

Participants enter a paired test only with both sessions. Epoch counts are reported, not equalised between sessions.

In [4]:
p = tables['stats_participants']
display(p.groupby(['analysis', 'eyes'])['included'].agg(included='sum', listed='size'))
p[~p['included']]

,,included,listed
analysis,eyes,,
primary,open,68,71
secondary,closed,36,38


,eyes,analysis,sub,included,reason
0,open,primary,sub-01,False,SD: cannot read recording: Archivo EEG incompl...
27,open,primary,sub-28,False,SD: cannot read recording: Archivo EEG incompl...
43,open,primary,sub-44,False,SD: cannot read recording: Archivo EEG incompl...
73,closed,secondary,sub-03,False,SD: cannot read recording: Archivo EEG incompl...
98,closed,secondary,sub-28,False,NS: cannot read recording: Archivo EEG incompl...


In [5]:
tables['stats_epochs']

,eyes,analysis,condition,n_recordings,epochs_used_mean,epochs_used_median,epochs_used_min,epochs_rejected_mean,paired_n,paired_mean_diff_sd_minus_ns,paired_p_wilcoxon
0,open,primary,NS,71,40.41,40,17,19.44,68,-4.353,4.773e-05
1,open,primary,SD,68,36.04,36,17,23.53,68,-4.353,4.773e-05
2,closed,secondary,NS,37,43.54,45,23,16.46,36,-3,0.2011
3,closed,secondary,SD,37,40.54,40,24,19.46,36,-3,0.2011


## Band power by region (log10 µV²), SD − NS

`mean_diff` and its 95 % CI are in log10 units (0.1 ≈ +26 % power). Wilcoxon is the main test, the paired t-test a sensitivity check. FDR (Benjamini–Hochberg) is applied across the 4 bands × 5 regions within each eye condition.

In [6]:
cols = ['band', 'region', 'n', 'mean_diff', 'ci95_low', 'ci95_high', 'cohens_dz', 'rank_biserial',
        'p_wilcoxon', 'p_fdr_wilcoxon', 'p_ttest', 'p_fdr_ttest']
log_power = tables['stats_region_log_power']
log_power[log_power['eyes'] == 'open'][cols]

,band,region,n,mean_diff,ci95_low,ci95_high,cohens_dz,rank_biserial,p_wilcoxon,p_fdr_wilcoxon,p_ttest,p_fdr_ttest
0,delta,frontal,57,0.1202,0.06635,0.1741,0.592,0.6031,7.473e-05,0.0002491,3.87e-05,0.0001548
1,delta,central,68,0.04959,-0.002623,0.1018,0.2299,0.3257,0.01959,0.02448,0.06231,0.07788
2,delta,temporal,63,0.08843,0.03781,0.139,0.44,0.4286,0.003101,0.004771,0.000889,0.001976
3,delta,parietal,65,0.1099,0.06102,0.1588,0.557,0.6168,1.54e-05,6.161e-05,3.033e-05,0.0001516
4,delta,occipital,62,0.1634,0.1023,0.2246,0.6788,0.703,1.486e-06,9.339e-06,1.427e-06,9.514e-06
5,theta,frontal,57,0.06751,0.02289,0.1121,0.4015,0.4676,0.002135,0.003826,0.003686,0.005464
6,theta,central,68,0.05445,0.0175,0.09139,0.3567,0.4254,0.002296,0.003826,0.00448,0.005974
7,theta,temporal,63,0.1105,0.0576,0.1634,0.5261,0.7202,6.686e-07,8.731e-06,9.451e-05,0.00027
8,theta,parietal,65,0.1136,0.07335,0.1538,0.6993,0.6802,1.868e-06,9.339e-06,4.17e-07,7.041e-06
9,theta,occipital,62,0.1591,0.1016,0.2166,0.7025,0.7184,8.731e-07,8.731e-06,7.041e-07,7.041e-06


In [7]:
log_power[log_power['eyes'] == 'closed'][cols]

,band,region,n,mean_diff,ci95_low,ci95_high,cohens_dz,rank_biserial,p_wilcoxon,p_fdr_wilcoxon,p_ttest,p_fdr_ttest
20,delta,frontal,36,0.05754,0.002077,0.113,0.351,0.3453,0.07175,0.1025,0.04244,0.06528
21,delta,central,36,0.1454,0.06775,0.2231,0.6335,0.6216,0.0007638,0.001697,0.000553,0.001229
22,delta,temporal,36,0.05689,-0.004515,0.1183,0.3135,0.4114,0.0308,0.04738,0.06834,0.09762
23,delta,parietal,34,0.1565,0.09367,0.2194,0.869,0.7983,1.126e-05,3.755e-05,1.513e-05,4.641e-05
24,delta,occipital,33,0.1173,0.0488,0.1857,0.6074,0.6328,0.001025,0.002006,0.001434,0.002869
25,theta,frontal,36,0.1578,0.09968,0.2159,0.9188,0.8318,1.834e-06,1.223e-05,3.39e-06,1.695e-05
26,theta,central,36,0.2163,0.1471,0.2855,1.058,0.9129,5.061e-08,1.012e-06,2.711e-07,5.422e-06
27,theta,temporal,36,0.1522,0.09953,0.2049,0.9774,0.8438,1.154e-06,1.154e-05,1.166e-06,1.166e-05
28,theta,parietal,34,0.1805,0.117,0.244,0.9921,0.842,2.499e-06,1.249e-05,1.819e-06,1.213e-05
29,theta,occipital,33,0.17,0.1017,0.2383,0.8825,0.7683,3.766e-05,0.0001076,1.624e-05,4.641e-05


### Relative power (%), SD − NS

In [8]:
rel = tables['stats_region_rel_power']
rel[cols[:2] + ['eyes'] + cols[2:]].sort_values(['eyes', 'band', 'region'])

,band,region,eyes,n,mean_diff,ci95_low,ci95_high,cohens_dz,rank_biserial,p_wilcoxon,p_fdr_wilcoxon,p_ttest,p_fdr_ttest
31,alpha,central,closed,36,-5.823,-10.18,-1.47,-0.4526,-0.4565,0.016,0.03201,0.01021,0.02042
30,alpha,frontal,closed,36,-2.595,-5.111,-0.07908,-0.349,-0.3213,0.09458,0.1505,0.04358,0.07923
34,alpha,occipital,closed,33,-13.16,-19.62,-6.709,-0.7232,-0.8075,1.145e-05,3.816e-05,0.0002264,0.0006468
33,alpha,parietal,closed,34,-13.09,-19.22,-6.958,-0.745,-0.8017,1.01e-05,3.816e-05,0.0001255,0.0004446
32,alpha,temporal,closed,36,-4.039,-8.54,0.4618,-0.3036,-0.3814,0.04597,0.08359,0.07704,0.1284
36,beta,central,closed,36,-0.307,-1.626,1.012,-0.07873,0.006006,0.9814,0.9814,0.6396,0.7107
35,beta,frontal,closed,36,1.183,-0.2618,2.628,0.2771,0.3183,0.0978,0.1505,0.1054,0.1621
39,beta,occipital,closed,33,-0.2125,-1.936,1.511,-0.04372,0.05169,0.805,0.8474,0.8033,0.8033
38,beta,parietal,closed,34,-0.6742,-1.877,0.5288,-0.1955,-0.1328,0.5097,0.5997,0.2624,0.3354
37,beta,temporal,closed,36,-0.4789,-2.407,1.45,-0.08403,-0.1562,0.423,0.564,0.6173,0.7107


## Frontal theta/beta ratio (mean log10 ratio over frontal channels)

In [9]:
tables['stats_frontal_theta_beta']

,eyes,analysis,measure,n,mean_ns,mean_sd,mean_diff,median_diff,ci95_low,ci95_high,cohens_dz,rank_biserial,wilcoxon_W,p_wilcoxon,t,p_ttest
0,open,primary,log10_frontal_theta_beta,57,0.1598,0.1472,-0.01256,-0.01049,-0.07163,0.04651,-0.05642,-0.06957,769,0.6478,-0.4259,0.6718
1,closed,secondary,log10_frontal_theta_beta,36,0.1599,0.2005,0.04066,0.0225,-0.02476,0.1061,0.2103,0.1772,274,0.3624,1.262,0.2154


## Session-order check

SD − NS difference in participants who did NS first vs SD first (Mann–Whitney U, FDR within each eye condition).

In [10]:
order = tables['stats_session_order']
order.sort_values('p_mannwhitney').head(10)

,eyes,analysis,measure,band,region,n_ns_first,n_sd_first,median_diff_ns_first,median_diff_sd_first,U,p_mannwhitney,p_fdr_mannwhitney
36,closed,secondary,log10_power,beta,frontal,17,19,0.02423,0.1175,104,0.07089,0.7521
29,closed,secondary,log10_power,theta,parietal,15,19,0.1017,0.2116,99,0.1358,0.7521
17,open,primary,log10_power,beta,temporal,36,27,-0.007283,0.06427,384,0.1586,0.8639
21,closed,secondary,log10_power,delta,frontal,17,19,0.07937,-0.02728,203,0.1939,0.7521
30,closed,secondary,log10_power,theta,occipital,14,19,0.1109,0.1698,99,0.2224,0.7521
22,closed,secondary,log10_power,delta,central,17,19,0.1836,0.07823,199,0.241,0.7521
32,closed,secondary,log10_power,alpha,central,17,19,0.05696,0.02504,194,0.3106,0.7521
1,open,primary,log10_power,delta,central,40,28,0.1126,0.0307,638,0.3342,0.8639
38,closed,secondary,log10_power,beta,temporal,17,19,0.05509,0.06053,131,0.3418,0.7521
0,open,primary,log10_power,delta,frontal,35,22,0.1353,0.06145,443,0.3459,0.8639


## Per-electrode cluster-based permutation test

SD − NS log10 power per electrode, `mne.stats.permutation_cluster_1samp_test`, adjacency from the dataset's `electrodes.tsv`. Bad channels are set to a 0 difference (conservative); see `n_imputed_zero` in `stats_cluster_channels.csv`.

In [11]:
clusters = tables['cluster_results']
clusters[clusters['significant']][['eyes', 'band', 'sign', 'n_channels', 't_sum', 'p_cluster',
                                   'n_subjects', 'channels']]

,eyes,band,sign,n_channels,t_sum,p_cluster,n_subjects,channels
0,open,delta,positive (SD > NS),10,34.1,0.0174,68,AF3;AF4;Fz;F1;F2;F3;F4;F5;F6;F8
1,open,delta,positive (SD > NS),30,105.6,0.0012,68,C3;C6;CP1;CP2;CP3;CP4;CP5;CP6;T8;TP7;TP8;TP9;T...
3,open,theta,positive (SD > NS),42,166.7,0.0002,68,F2;F4;F6;F7;F8;FC5;FC6;C3;C4;C5;C6;CP1;CP2;CP3...
5,open,alpha,positive (SD > NS),17,46.82,0.0028,68,T8;TP10;P1;P2;P3;P4;P5;P6;P7;P8;POz;PO3;PO4;PO...
7,open,beta,positive (SD > NS),9,27.37,0.0066,68,AF3;F1;F3;F5;F7;FC5;C3;C5;FT7
8,open,beta,positive (SD > NS),22,64.64,0.0002,68,F2;F4;F6;F8;FC6;CP1;FT8;TP10;Pz;P1;P2;P3;P4;P5...
9,closed,delta,positive (SD > NS),41,140.3,0.0004,36,AF3;Fz;F1;F2;F3;FC2;FC3;FC4;FC5;FC6;Cz;C1;C2;C...
10,closed,theta,positive (SD > NS),54,259.8,0.0002,36,Fp1;Fpz;Fp2;AF3;AF4;AF7;Fz;F1;F2;F3;F4;F5;F6;F...
17,closed,alpha,negative (SD < NS),8,-22.87,0.0412,36,Pz;P4;P8;POz;PO4;PO8;O1;O2
19,closed,beta,positive (SD > NS),21,79.12,0.002,36,Fp1;Fpz;AF3;Fz;F1;F2;F3;F8;FC2;FC4;FC6;Cz;C2;C...
